# IOAI — 2025 Summer National Parametric Palette (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('petike.pth 는 노트북 첫 셀이 gdown 으로 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Parametric Palette — 모범답안 ("Magic" latent optimisation)

> **HAIO 2025 — Summer National Finals (CV / generative)**

The frozen VAE never saw magenta/orange, so *encoding* a target image can't reproduce those colours. Instead we **optimise the latent `z` by gradient descent** so the frozen decoder's output matches the target — exactly the Madarian-Cow "Magic" mechanism. The decoder + the built-in shape classifier are differentiable, so we can backprop the colour/shape objective into `z`.

In [ ]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw
# petike.pth: 헤드리스에서 gdown 자동 다운로드(없으면)
if not os.path.exists('data/petike.pth') and not os.path.exists('petike.pth'):
    os.makedirs('data', exist_ok=True)
    try:
        import gdown
    except ImportError:
        import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','gdown'],check=True); import gdown
    gdown.download(id='1wuYIUV9rp-OeSWDSrTllmrTt50YzbxxP', output='data/petike.pth', quiet=True)
PETIKE='data/petike.pth' if os.path.exists('data/petike.pth') else 'petike.pth'
class Petike(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv2d(3,32,4,2,1),nn.BatchNorm2d(32),nn.ReLU(),nn.Conv2d(32,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.ReLU())
        self.flatten=nn.Flatten(); self.fc_mu=nn.Linear(128*4*4,48); self.fc_logvar=nn.Linear(128*4*4,48)
        self.decoder_input=nn.Linear(48,48)
        self.decoder=nn.Sequential(nn.Unflatten(1,(3,4,4)),nn.ConvTranspose2d(3,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(),nn.ConvTranspose2d(64,32,4,2,1),nn.BatchNorm2d(32),nn.ReLU(),nn.ConvTranspose2d(32,16,4,2,1),nn.BatchNorm2d(16),nn.ReLU(),nn.Conv2d(16,3,3,padding=1),nn.Sigmoid())
        self.register_buffer('shape_input',torch.zeros(48)); self.shape_input[0:24]=1.0
        self.register_buffer('color_input',torch.zeros(48)); self.color_input[24:48]=1.0
        for i in [0,12,24,36]: self.color_input[i]=0.0; self.shape_input[i]=0.0
        self.shape_classifier=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,3))
        self.color_classifier=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,3))
        self.size_regressor=nn.Sequential(nn.Linear(48,16),nn.ReLU(),nn.Linear(16,1))
        self.mask_head=nn.Sequential(nn.Linear(48,48),nn.ReLU(),nn.Linear(48,32*32),nn.Sigmoid())
    def encode(self,x): return self.fc_mu(self.flatten(self.encoder(x)))
    def decode(self,z): return self.decoder(self.decoder_input(z))
m=Petike(); m.load_state_dict(torch.load(PETIKE,map_location='cpu')); m.eval()
for p in m.parameters(): p.requires_grad_(False)
def draw(shape,color=(255,0,0)):
    img=Image.new('RGB',(32,32),(255,255,255)); d=ImageDraw.Draw(img)
    if shape=='circle': d.ellipse([8,8,24,24],fill=color)
    elif shape=='square': d.rectangle([9,9,23,23],fill=color)
    else: d.polygon([(16,7),(7,25),(25,25)],fill=color)
    return torch.tensor(np.array(img)/255.,dtype=torch.float).permute(2,0,1)[None]
sidx={sh:int(m.shape_classifier(m.encode(draw(sh))*m.shape_input).argmax()) for sh in ['triangle','square','circle']}
print('petike loaded · shape indices:', sidx)

In [ ]:
def sat(img): return (img.max(1).values-img.min(1).values)[0]
def magenta(img):
    R,G,B=img[:,0],img[:,1],img[:,2]; return (F.relu(R-G)*F.relu(B-G))[0].sum()/(sat(img).sum()+1e-6)
def orange(img):
    s=sat(img); w=s/(s.sum()+1e-6); mc=(img[0]*w).sum(dim=(1,2)); b=torch.tensor([1.,0.5,0.])
    return F.relu((mc*b).sum()/(mc.norm()*b.norm()+1e-6))
def blue(img):
    R,G,B=img[:,0],img[:,1],img[:,2]; return F.relu(B-torch.maximum(R,G)).mean()
def sp(z,i): return F.softmax(m.shape_classifier(z*m.shape_input),1)[0,i]
def optimise(kind, seed):
    torch.manual_seed(seed)
    init={'mc':('circle',(255,0,0)),'ot':('triangle',(255,0,0)),'mb':('square',(0,255,0))}[kind]
    z=(m.encode(draw(*init)).detach()+0.05*torch.randn(1,48)).requires_grad_(True)
    opt=torch.optim.Adam([z],lr=0.05)
    for _ in range(250):
        img=m.decode(z)
        if kind=='mb': loss=blue(img)
        elif kind=='mc': loss=-(2*magenta(img)+0.5*torch.log(sp(z,sidx['circle'])+1e-6))
        else: loss=-(2*orange(img)+0.5*torch.log(sp(z,sidx['triangle'])+1e-6))
        opt.zero_grad(); loss.backward(); opt.step()
    return z.detach()
kinds=['mc']*5+['ot']*5+['mb']*5
Z=torch.cat([optimise(k, s) for s,k in enumerate(kinds)],0).numpy().astype('float32')
np.save('submission.npy', Z)
print('wrote submission.npy', Z.shape)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)